# Token guesses at random initialization**Question.** At random initialization, how do the distributions of the model's tokenguesses compare with the empirical token distribution of the corpus — in overallconcentration and token by token — and how stable is that comparison acrossindependent model initializations?This notebook is a reading surface, not an implementation. Every number and every figurecomes from `llm_behavior_lab.analysis`, so anything shown here can be recomputed, tested,and reused elsewhere. To extend the study, add the measure to`llm_behavior_lab.analysis.aggregation` (with a test) and call it from a new section below.> The model is **untrained**. Nothing here describes model quality. These are properties of> a randomly initialized network and of the corpus it is being compared against.

## Experimental setupRun the experiment first; this notebook only reads its output.```bash# smoke check on the tracked fixture (fast, not scientifically meaningful)python3 scripts/run_initialization_distribution_experiment.py \  --data-config configs/data/tiny_text.yaml \  --num-initializations 3 --num-windows 16 --block-size 16 --num-replicates 2# first meaningful experimentpython3 scripts/run_initialization_distribution_experiment.py \  --data-config configs/data/wikitext2.yaml --offline```Everything except the model-initialization seed is held fixed: corpus, tokenizer,vocabulary, split, and the evaluation positions themselves. The positions are chosendeterministically before any model exists, so no difference between initializations cancome from looking at different text.

In [ ]:
from pathlib import Pathfrom llm_behavior_lab.analysis import (    load_record,    sampling_adequacy,    summarize_policy,    within_initialization_sampling_spread,)from llm_behavior_lab.analysis.figures import generate_all_figures# Point this at the run you want to read.RUN_DIR = sorted(Path("../outputs/initialization_distribution").glob("*"))[-1]FIGURE_DIR = RUN_DIR / "figures"record = load_record(RUN_DIR / "analyses")analysis = record.metadata["analysis"]record.num_initializations, record.num_replicates, record.vocab_size

## Dataset, model, and protocolThe record carries its own provenance, so a figure can never be read without the settingsthat produced it.

In [ ]:
print("dataset      :", record.metadata["dataset"].get("name"), "via", record.metadata["dataset"].get("route"))print("model        :", record.metadata["model_name"], f'({record.metadata["model_parameter_count"]} parameters)')print("tokenizer    :", record.metadata["tokenizer"])print("split        :", analysis["split"], f'({analysis["split_token_count"]} tokens)')print("positions    :", analysis["num_positions"], analysis["evaluation_positions"])print("model seeds  :", analysis["model_seeds"])print("sampling     :", analysis["sampling"])

## Initialization and sampling protocolTwo sources of randomness are deliberately separated.* **Model initialization** — one seed per initialization, the independent unit for every  standard error reported below.* **Token sampling** — only affects the stochastic policy. Each initialization produces one  set of logits, and the nucleus policy is replicated several times from those same logits.  Replicates are averaged *within* an initialization before initializations are compared,  so sampling noise is never reported as initialization noise.`within_initialization_sampling_spread` reports the sampling noise on its own, next to thebetween-initialization spread of the same quantity.

In [ ]:
within_initialization_sampling_spread(record)

## Empirical sampling adequacy (figure 0)Before comparing any model against the corpus, check that the analyzed positions representthe corpus at all. This compares the ranked token distribution of the whole split with theranked distribution of the next-token targets at the evaluated positions.This is a statement about the **corpus sample only** — no model is involved. If the twocurves separate visibly, increase `--num-windows`; adding initializations cannot repair anunrepresentative position set.

In [ ]:
figures = generate_all_figures(record, FIGURE_DIR)sampling_adequacy(record)

## Ranked frequency profiles (figure 1)Each distribution is ranked independently, so rank *r* of the guess curve and rank *r* ofthe corpus curve are generally **different tokens**. The comparison is therefore about howconcentrated the distributions are, not about which tokens agree.The corpus curve carries no error band: it is one fixed distribution, not a sample overinitializations.

In [ ]:
summaries = {policy: summarize_policy(record, policy) for policy in ("greedy", "nucleus")}{policy: summary.as_dict() for policy, summary in summaries.items()}

## Token-wise mismatch (figure 2)Here token identity is preserved. The gap `|q(s,i) - p(i)|` is taken between the **sametoken ID** in both distributions, and only then ranked. Ranking first would compareunrelated tokens and report a far smaller, misleading mismatch.Two curves per policy:* **typical** — the mean over initializations of each initialization's ranked gap profile;* **persistent** — the ranked gaps of the initialization-averaged guess distribution,  `|E_s[q(s,i)] - p(i)|`.Large typical with small persistent means initializations disagree about which tokens theyover-select and the excess averages away. Both large means the same tokens are favouredevery time — a systematic property of the architecture and initialization scheme.

## Complementary diagnostics (figure 3)Both ranked figures discard token identity at some point. The scatter keeps it throughout:one marker per token, corpus fraction against mean guess fraction, with the identity linefor reference. Markers above the line are over-selected relative to how often the tokenactually occurs.Scalar summaries reported alongside: total variation distance, Jensen–Shannon divergence,entropy, effective support `exp(H)`, zero-guess token count, and the top-1/top-2concentration gap.

In [ ]:
for path in figures:    print(path)

## Observations and caveatsRecord observations from **your** run here rather than relying on any pre-written text.Caveats that always apply:* The model is untrained. None of this measures quality.* Zero-guess counts depend strongly on the number of evaluated positions. With fewer  positions than vocabulary entries most tokens are unreachable regardless of the model.* The **mean predicted probability** vector and the **selected-guess frequency** vector are  different quantities. The record stores both and never mixes them.* Standard errors are across model initializations only. They say nothing about whether the  evaluated positions represent the corpus — figure 0 answers that separately.* Nucleus results depend on the temperature and top-p recorded in the metadata above.  Changing them changes the policy, not just the noise.